# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器

本筆記本專為 **Google Colab VS Code Extension** 設計：
- 在本地 VS Code 編輯代碼
- 運算核心切換為 **Google Colab 遠端運算**（本機 CPU 使用率 0%）
- 調參目標：**LMR 第一階段核心六軸（BASE_OFFSET + HIST_SCALE + CUTNODE + TTCAP + MOVECOUNT + TABLE_SCALE）深度調參（80 輪 x 150 局 = 12,000 局）**

### 💡 在 VS Code 中連接 Colab 核心步驟：
1. 點擊本編輯器右上角的 **「Select Kernel（選取核心）」**
2. 選擇 **「Google Colab」**
3. 連接成功後即可像執行本機代碼一樣，直接點擊下方儲存格依序執行！


In [ ]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python --version

In [ ]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout {branch}
!git pull origin {branch}
!pip install -q numba chess numpy

In [ ]:
# 3. 測試環境與常數載入
import sys
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

target_params = [
    "LMR_BASE_OFFSET",
    "LMR_HISTORY_SCALE",
    "LMR_CUTNODE_BONUS",
    "LMR_TTCAPTURE_BONUS",
    "LMR_MOVECOUNT_FACTOR",
    "LMR_TABLE_SCALE_PERCENT"
]
print("✅ 成功載入！LMR 第一階段核心六軸基準預設值：")
print(get_default_params(names=target_params))


In [ ]:
# 4. 啟動 L1 SPSA LMR 第一階段核心六軸雲端對抗調參（目標 80 輪 x 150 局 = 12,000 局）
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/lmr_p1_deep_l1.jsonl"
state_file = "tune_search/states/lmr_p1_deep_l1.json"
total_campaign_iters = 80

# 守護機制：若處於全新虛擬機（情況 B：無進度存檔），自動還原前 28 輪（4,200 局）進度！
if not os.path.exists(state_file):
    try:
        import tools.recover_lmr_p1_iter28 as recover
        recover.restore_checkpoint(concurrency=concurrency)
    except Exception as e:
        print(f"⚠️ 自動還原腳本提示: {e}")

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params LMR_BASE_OFFSET,LMR_HISTORY_SCALE,LMR_CUTNODE_BONUS,LMR_TTCAPTURE_BONUS,LMR_MOVECOUNT_FACTOR,LMR_TABLE_SCALE_PERCENT \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 150 \
      --nodes 50000 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ 80 輪調參已全部完成！請直接執行 Cell 5 產出候選參數。")


In [ ]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後半段平均產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/lmr_p1_deep_l1.jsonl --tail 0.5 --out tune_search/candidate_lmr_p1_deep.json
print("\n=== 候選參數 JSON 內容 ===")
!cat tune_search/candidate_lmr_p1_deep.json


In [ ]:
# 6. 直接在雲端進行 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p1_deep.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_lmr_p1_deep.json
